# Response API Tool Calls

* [/responses](https://docs.litellm.ai/docs/response_api)

> LiteLLM provides an endpoint in the spec of [OpenAI's /responses API](https://docs.litellm.ai/docs/response_api).


* [LiteLLM - Getting Started](https://docs.litellm.ai/docs/)
* [LiteLLM Cookbook](https://github.com/BerriAI/litellm/tree/main/cookbook)

# Setup

In [1]:
%%html
<style>
table {float:left}
</style>

In [2]:
import os
import json
import operator
from datetime import datetime, timedelta, timezone
from typing import (
    List, Dict, Any, Literal, Optional, Callable, Annotated
)

import regex as re
from pydantic import BaseModel, Field, ConfigDict

from openai import OpenAI
from tavily import TavilyClient
from langgraph.graph import StateGraph, START, END
import litellm
import mdformat
import trafilatura
from IPython.display import Markdown, display

## API Keys

In [3]:
path_to_openai_key:str = os.path.expanduser('~/.openai/api_key')
with open(path_to_openai_key, 'r', encoding='utf-8') as file:
    os.environ["OPENAI_API_KEY"] = file.read().strip()

path_to_tavily_key:str = os.path.expanduser('~/.tavily/api_key')
with open(path_to_tavily_key, 'r', encoding='utf-8') as file:
    os.environ["TAVILY_API_KEY"] = file.read().strip()

## Models

In [4]:
MODEL: str = "gpt-5.2"   # "openai/gpt-4o" 

---
# Tool Calling Protocol


## OpenAI API Specification


OpenAI defined the **Function (Tool) Calling Protocol** as in [Function calling](https://developers.openai.com/api/docs/guides/function-calling). 

  1. LLM responds with role: "assistant" message containing tool_calls (with id, function.name, function.arguments)
  2. You execute the tools                                                      
  3. You send back one message per tool call with role: "tool", tool_call_id matching the original id, and content containing the result as a string

There are six stages in the Protocol.

1. Tool Definition by Application and Declaration to LLM at Prompt Message
> When we make an API request to the model with a prompt, we can include a list of tools the model could consider using.

2. Tool Call Decision by LLM  (**Reasoing** and **Decisoning**)
> tool call refers to a special kind of decision response from LLM to call one of the tools to execute the prompt given.
   
3. Tool Execution by Application
4. Tool Output Usage by LLM
> tool call output refers to the response a tool generates using the input from a model’s tool call. We send **all of the tool definition, the original prompt, the model’s tool call, and the tool call output back** to LLM to finally receive a text response.

5. LLM completes the prompt

### Tool Calling Workflow

```
1. Application to LLM
   │ Define and Declare Tools in the prompt
   ▼
2. LLM
   │ Reason/Decide Tool Calls
   ▼
3. Application/Orchestration
   │ Tool Executions
   ▼
4. LLM Receives Tool Outputs
   │ Use Tool Outputs
   ▼
5. LLM Complete the prompt
```

<img src="../image/tool_call_flow.png" align="left" width=500/>



## LiteLLM

Use LiteLLM as the wrapper for the OpenAI Response API for Function calling.

* [LiteLLM - Response API](https://docs.litellm.ai/docs/providers/openai/responses_api)
* [LiteLLM - Function Calling](https://docs.litellm.ai/docs/completion/function_call)

1. `litellm.responses()` with `tools=` and `instructions=`
2. Parse `response.output` for `type=="function_call"` items and execute functions
3. Second `litellm.responses()` call with tool results appended to `input`


## Tool Definition

* [Defining Functions (Tools)](https://developers.openai.com/api/docs/guides/function-calling#defining-functions)

First, you define a tool based on the OpenAI definition. This is what LiteLLM accepts as well.

| Field       | Description                                          |
|-------------|------------------------------------------------------|
| type        | This should always be function                       |
| name        | The function’s name (e.g. get_weather)               |
| description | Details on when and how to use the function          |
| parameters  | JSON schema defining the function’s input arguments  |
| strict      | Whether to enforce strict mode for the function call |


### Tool Definition Format

The tool definition format differs between Chat Completion and Response API:

**Response API** — flat, no nested `"function"` wrapper:
```
{
  "type": "function",
  "name": "...",             ← promoted to top level
  "description": "...",
  "parameters": { JSON schema }
}
```

* [Response API - Function Calling](https://developers.openai.com/api/docs/guides/function-calling)

**Example (Response API):**
```
{
  "type": "function",
  "name": "get_weather",
  "description": "Retrieves current weather for the given location.",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string",
        "description": "City and country e.g. Bogota, Colombia"
      },
      "units": {
        "type": "string",
        "enum": ["celsius", "fahrenheit"],
        "description": "Units the temperature will be returned in."
      }
    },
    "required": ["location", "units"],
    "additionalProperties": false
  },
  "strict": true
}
```

### Namespace

* [Defining namespace](https://developers.openai.com/api/docs/guides/function-calling#defining-namespaces)

> Namespaces help organize similar tools and are especially useful when the model must choose between tools that serve different systems or purposes.

```
{
  "type": "namespace",
  "name": "crm",
  "description": "CRM tools for customer lookup and order management.",
  "tools": [
      tool_definition+
 ]
}
```


### Tool Parameter JSON Schema

[Pydantic JSON Schema](https://docs.pydantic.dev/latest/concepts/json_schema/) to generate the JSON Schema for the Tool Definition. See [Tavilty Search](https://docs.tavily.com/sdk/python/reference#tavily-search) for the tool parameters.


#### Tool 

In [5]:
# Function for the Search Tool
search_tool: Callable = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
if not hasattr(search_tool, "tool_name"):
    search_tool.name = "search_tool"

name_to_tool: Dict[str, Callable] = {
    # search_tool.name: search_tool.search
    search_tool.name: search_tool.search
}

#### Tool Parameters

Use Pydantic to generate the JSON Schema for the parameters. See [tavily search](https://docs.tavily.com/sdk/python/reference#tavily-search) for the prameter details.



In [6]:
# Parameters for the Search Tool (Tavily)
# https://docs.tavily.com/sdk/python/reference#tavily-search
class SearchToolParameters(BaseModel):
    """
    Search the web for current events, news, or deep research.
    """
    query: str = Field(description="The search query to look up")
    
    topic: Literal["general", "news", "finance"] = Field(
        default="general",
        description="Category of search. Use 'news' for current events/politics, 'finance' for market data."
    )
    search_depth: Literal["basic", "advanced"] = Field(
        default="basic",
        description="Use 'basic' for quick facts. Use 'advanced' for complex queries needing more context."
    )
    time_range: Optional[Literal["day", "week", "month", "year"]] = Field(
        default=None,
        description="Filter results by publication date. Especially useful with topic='news'."
    )
    max_results: int = Field(
        default=5, ge=1, le=10,
        description="Number of search results to return."
    )
    model_config = ConfigDict(extra="forbid")  # adds additionalProperties: false

In [7]:
search_tool_parameters: Dict[str, Any] = SearchToolParameters.model_json_schema()
print(json.dumps(search_tool_parameters, indent=2, default=str))

{
  "additionalProperties": false,
  "description": "Search the web for current events, news, or deep research.",
  "properties": {
    "query": {
      "description": "The search query to look up",
      "title": "Query",
      "type": "string"
    },
    "topic": {
      "default": "general",
      "description": "Category of search. Use 'news' for current events/politics, 'finance' for market data.",
      "enum": [
        "general",
        "news",
        "finance"
      ],
      "title": "Topic",
      "type": "string"
    },
    "search_depth": {
      "default": "basic",
      "description": "Use 'basic' for quick facts. Use 'advanced' for complex queries needing more context.",
      "enum": [
        "basic",
        "advanced"
      ],
      "title": "Search Depth",
      "type": "string"
    },
    "time_range": {
      "anyOf": [
        {
          "enum": [
            "day",
            "week",
            "month",
            "year"
          ],
          "type": "str

#### Tool Description

In [8]:
search_tool_description: str = SearchToolParameters.__doc__.strip()
print(search_tool_description)

Search the web for current events, news, or deep research.


### Tool Definition of the Tabiliy Web Search

In [9]:
# Response API: flat tool definition (name/description/parameters at top level)
# Ref: https://developers.openai.com/api/docs/guides/function-calling
search_tool_definition = {
    "type": "function",
    "name": search_tool.name,
    "description": search_tool_description,
    "parameters": search_tool_parameters,
    # Open AI document example sets "strict": True but causes BadRequestError.
    # See developers.openai.com/api/docs/guides/tools?tool-type=function-calling
    # BadRequestError: Error code: 400 - {'error': {
    #   'message': "Invalid schema for function 'search_tool': In context=(), 
    #              'required' is required to be supplied and to be an array 
    #              including every key in properties. Missing 'topic'.", 
    #              'type': 'invalid_request_error', 
    #              'param': 'tools[0].parameters', 
    #              'code': 'invalid_function_parameters'
    # }}
    # "strict": True
}
print(json.dumps(search_tool_definition, indent=2, default=str))

{
  "type": "function",
  "name": "search_tool",
  "description": "Search the web for current events, news, or deep research.",
  "parameters": {
    "additionalProperties": false,
    "description": "Search the web for current events, news, or deep research.",
    "properties": {
      "query": {
        "description": "The search query to look up",
        "title": "Query",
        "type": "string"
      },
      "topic": {
        "default": "general",
        "description": "Category of search. Use 'news' for current events/politics, 'finance' for market data.",
        "enum": [
          "general",
          "news",
          "finance"
        ],
        "title": "Topic",
        "type": "string"
      },
      "search_depth": {
        "default": "basic",
        "description": "Use 'basic' for quick facts. Use 'advanced' for complex queries needing more context.",
        "enum": [
          "basic",
          "advanced"
        ],
        "title": "Search Depth",
        "type

## Tool Declaration

You declare the **tool definitions** to the LLM.

**Response API** — flat format:
```
tools = [
    {
      "type": "function",
      "name": "SearchTool",
      "description": "...",
      "parameters": { JSON Schema }
    },
    ...
]
```

**Chat Completion (deprecated)** — nested `"function"` wrapper:
```
tools = [
    {
      "type": "function",
      "function": {
        "name": "SearchTool",
        "description": "...",
        "parameters": { JSON Schema }
      }
    },
    ...
]
```

* [LiteLLM - Response API](https://docs.litellm.ai/docs/providers/openai/responses_api)

```python
# Response API
response = litellm.responses(
    model=model,
    input=input,
    tools=tools,            # <- declare tools here (flat format)
    instructions=instructions,
)
```

*Example input:*
```
{
  "model": "openai/gpt-4o",
  "input": [
    {
        "role": "user",
        "content": "Weather in Tokyo?"
    }
  ],
  "tools": [
    {
      "type": "function",
      "name": "get_weather",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {"type": "string"}
        }
      }
    }
  ]
}
```


### Tool Declartion for the Search

In [10]:
tool_declarations: List[Dict[str, Any]] = [
    search_tool_definition
]
tool_declarations

[{'type': 'function',
  'name': 'search_tool',
  'description': 'Search the web for current events, news, or deep research.',
  'parameters': {'additionalProperties': False,
   'description': 'Search the web for current events, news, or deep research.',
   'properties': {'query': {'description': 'The search query to look up',
     'title': 'Query',
     'type': 'string'},
    'topic': {'default': 'general',
     'description': "Category of search. Use 'news' for current events/politics, 'finance' for market data.",
     'enum': ['general', 'news', 'finance'],
     'title': 'Topic',
     'type': 'string'},
    'search_depth': {'default': 'basic',
     'description': "Use 'basic' for quick facts. Use 'advanced' for complex queries needing more context.",
     'enum': ['basic', 'advanced'],
     'title': 'Search Depth',
     'type': 'string'},
    'time_range': {'anyOf': [{'enum': ['day', 'week', 'month', 'year'],
       'type': 'string'},
      {'type': 'null'}],
     'default': None,
  

## LLM
### Input to LLM

Build the input items to LLM (the prompt).

* Response API: `input=` takes a list of message dicts `{role, content}`
* System instructions are passed separately as `instructions=`


In [11]:
# Response API: system instructions go in the instructions= parameter (not in input items)
# Chat Completion: {"role": "system", "content": "..."} was the first element of messages
# Ref: https://developers.openai.com/api/docs/guides/migrate-to-responses#system-and-developer-messages
SYSTEM_INSTRUCTIONS: str = """
You are a helpful assistant. Use tools when needed.
Verify the relevance of retrieved results before using them, and apply an intelligent
intent filter so you keep only the news items that align with the user inquiry.
Only report items that are relevant to the user request and supported by the tool output.
If the search results are noisy or insufficient, say that and do not infer missing facts.
Respnse follows the given Python JSON schema where:
1. news field is the news articies as list.
2. uri field is the URI of the news article found.
"""

# Response API: input= contains only user/assistant turn items (no system message)
# Chat Completion: messages= contained system + user messages together
input_prompt: List[Dict[str, str]] = [
    {
        "role": "user",      # <--- same role as before
        "content": "What are the top news headlines about Iran US conflicts?"
    }
]

### Output format from LLM


In [12]:
# RFC 3986 URI: scheme "://" followed by non-whitespace characters                                                                                                                                        
UriStr = Annotated[str, Field(pattern=r"^[a-zA-Z][a-zA-Z0-9+\-.]*://\S+$")]

class ResponseStructuredFormat(BaseModel):
    """Schema for the Python query answer"""
    news: list[str] = Field(
        description="List of news"
    )
    uri: list[UriStr] = Field(
        description="URIs of the specifications referred"
    )
    #--------------------------------------------------------------------------------
    # JSON Schema additionalProperty: False is required for the API to work
    # community.openai.com/t/schema-additionalproperties-must-be-false-when-strict-is-true/929996
    #--------------------------------------------------------------------------------
    # BadRequestError: Error code: 400 - {
    # {
    #   "message": "Invalid schema for response_format ... additionalProperties is required to be supplied and to be false.",
    #   "type": "invalid_request_error",
    #   "param": "text.format.schema",
    #   "code": "invalid_json_schema"
    # }
    #--------------------------------------------------------------------------------
    model_config = ConfigDict(extra="forbid")  # adds additionalProperties: false

ResponseStructuredFormat.model_json_schema()

{'additionalProperties': False,
 'description': 'Schema for the Python query answer',
 'properties': {'news': {'description': 'List of news',
   'items': {'type': 'string'},
   'title': 'News',
   'type': 'array'},
  'uri': {'description': 'URIs of the specifications referred',
   'items': {'pattern': '^[a-zA-Z][a-zA-Z0-9+\\-.]*://\\S+$',
    'type': 'string'},
   'title': 'Uri',
   'type': 'array'}},
 'required': ['news', 'uri'],
 'title': 'ResponseStructuredFormat',
 'type': 'object'}

### LLM Call

* [Function Calling](https://developers.openai.com/api/docs/guides/tools?tool-type=function-calling)

In [30]:
client = OpenAI()
response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_INSTRUCTIONS,
    input=input_prompt,   # plain text query can be used as well.
    tools=tool_declarations,
    tool_choice="auto",
    temperature=0,
    max_output_tokens=2048,
    text={
        "format": {
            # ----------------------------------------------------------------------
            # Structured Output (JSON Schema)
            # ----------------------------------------------------------------------
            "type": "json_schema",
            "name": "ResponseStructuredFormat",
            "strict": True,
            "schema": ResponseStructuredFormat.model_json_schema()
        }
    },
    stream=False
)

In [31]:
print(response.model_dump_json(indent=4))

{
    "id": "resp_06c103093fd0029b0069d314fa592c81938d3197f3d8ee1c8a",
    "created_at": 1775441146.0,
    "error": null,
    "incomplete_details": null,
    "instructions": "\nYou are a helpful assistant. Use tools when needed.\nVerify the relevance of retrieved results before using them, and apply an intelligent\nintent filter so you keep only the news items that align with the user inquiry.\nOnly report items that are relevant to the user request and supported by the tool output.\nIf the search results are noisy or insufficient, say that and do not infer missing facts.\nRespnse follows the given Python JSON schema where:\n1. news field is the news articies as list.\n2. uri field is the URI of the news article found.\n",
    "metadata": {},
    "model": "gpt-5.2-2025-12-11",
    "object": "response",
    "output": [
        {
            "arguments": "{\"query\":\"Iran US conflict top headlines\",\"topic\":\"news\",\"search_depth\":\"basic\",\"time_range\":\"week\",\"max_results\":10

## LLM Tool Calls

### Tool call handling logic

* [OpenAI - Handling function calls](https://developers.openai.com/api/docs/guides/function-calling#handling-function-calls)

**Response API** — tool calls are Items in `response.output` with `type == "function_call"`:

```python
for item in response.output:
    if item.type != "function_call":
        continue
    name = item.name
    args = json.loads(item.arguments)
    result = call_function(name, args)
    input_items.append({
        "type": "function_call_output",  # <- NOT role="tool"
        "call_id": item.id,              # <- NOT tool_call_id
        "output": str(result)            # <- NOT content
    })
```

### Tool Calls

**Response API**: the tool call is a top-level Item in `response.output`:

```
{
  "type": "function_call",
  "id": "call_JVHGxhAhxbmAlwiDkuc7fXk6",    <- call_id (used in function_call_output)
  "name": "search_tool",
  "arguments": "{\"query\":\"AI news\",\"topic\":\"news\",\"search_depth\":\"basic\"}"
}
```

In [32]:
# Response API: check response.output is non-empty
assert response.output, \
    f"Expected non-empty output:\n{json.dumps(response.model_dump(), indent=2, default=str)}"

In [33]:
def extract_tool_calls(response) -> List[Any]:
    """Extract function_call items from a Response API response.

    Response API: tool calls are Items in response.output with type == "function_call"
      Each item has: .type, .id (call_id), .name, .arguments (JSON string)
      Ref: https://developers.openai.com/api/docs/guides/function-calling#handling-function-calls

    Chat Completion equivalent:
      response.choices[0].message.tool_calls
      Each: .id, .function.name, .function.arguments
    """
    return [item for item in response.output if item.type == "function_call"]

In [34]:
def in_time_window(result: Dict[str, Any], query: str, now: datetime) -> bool:
    """Return True when the result was published within the last month."""
    del query  # Reserved for query-aware time parsing later.
    published_date = result.get("published_date")
    if not published_date:
        return False
    try:
        published_at = datetime.strptime(
            published_date, "%a, %d %b %Y %H:%M:%S %Z"
        ).replace(tzinfo=timezone.utc)
    except ValueError:
        return False
    return published_at >= (now - timedelta(days=30))

def is_bad_page_type(result: Dict[str, Any]) -> bool:
    """Return True for page types that are poor RAG evidence for this tutorial."""
    negative_markers = ('/event/', 'event', 'retreat', 'viewership', '/journalism/')
    title = str(result.get("title", "")).lower()
    url = str(result.get("url", "")).lower()
    return any(marker in title or marker in url for marker in negative_markers)

def keep_result(query: str, result: Dict[str, Any], now: datetime, threshold: float=0.70) -> bool:
    """Return True when a Tavily result is usable for the final answer."""
    print(json.dumps(result, indent=4, default=str))
    if not in_time_window(result, query, now):
        print("did not match time window")
        return False
    if is_bad_page_type(result):
        print("bad page type")
        return False
    if float(result.get("score", 0.0)) < threshold:
        print(f"no confidence < {threshold}")
        return False

    print("found")
    return True

def filter_search_results(results: List[Dict[str, Any]], query: str) -> List[Dict[str, Any]]:
    """Keep only search hits that are recent, relevant, and usable."""
    now = datetime.now(timezone.utc)
    return [result for result in results if keep_result(query, result, now)]

def format_search_result(result: Dict[str, Any]) -> str:
    """Serialize one Tavily result as JSON for the tool response."""
    payload = {
        "title": result.get("title", ""),
        "url": result.get("url", ""),
        "content": result.get("content", ""),
        "published_date": result.get("published_date"),
    }
    return json.dumps(payload, ensure_ascii=False)

def execute_tool_calls(tool_calls: List[Any]) -> List[Dict[str, Any]]:
    """Execute function_call items and return function_call_output items.

    Args:
        tool_calls: function_call items from response.output
                    Each has: .type, .id, .name, .arguments (JSON string)
    Returns:
        List of function_call_output dicts for the Response API input:
        [{"type": "function_call_output", "call_id": ..., "output": ...}]

    Response API output format:
        {"type": "function_call_output", "call_id": ..., "output": ...}
    Chat Completion equivalent:
        {"role": "tool", "tool_call_id": ..., "name": ..., "content": ...}
    Ref: https://developers.openai.com/api/docs/guides/function-calling#formatting-results
    """
    _tool_outputs: List[Dict[str, Any]] = []
    for tool_call in tool_calls:
        if tool_call.type != "function_call":
            continue

        func_name: str = tool_call.name                              # Chat Completion: tool_call.function.name
        func_args: Dict[str, Any] = json.loads(tool_call.arguments)  # Chat Completion: tool_call.function.arguments
        execution: Any = name_to_tool[func_name](**func_args)
        results = execution.get("results", [])
        if func_name == "search_tool":
            results = filter_search_results(results=results, query=func_args.get("query", ""))
        if not results:
            content = json.dumps({
                "query": func_args.get("query", ""),
                "results": [],
                "note": "No clearly relevant search results were found. Ask for a narrower query or different source."
            }, ensure_ascii=False)
        else:
            content = json.dumps({
                "query": func_args.get("query", ""),
                "results": [json.loads(format_search_result(r)) for r in results]
            }, ensure_ascii=False)

        _tool_outputs.append({
            "type": "function_call_output",  # Response API (Chat Completion: no "type", role="tool")
            "call_id": tool_call.call_id,     # Response API: call_id (NOT .id which is the item ID)
            # tool_call.id = item-level ID (e.g. fc_077c...)
            # tool_call.call_id = tool call reference ID (e.g. call_abc...) -- use this
            "output": content,               # Response API: output   (Chat Completion: content)
        })

    return _tool_outputs

In [35]:
import markdown
from html.parser import HTMLParser

class _StripHTML(HTMLParser):
    """Extract plain text from HTML, used as fallback in render_content."""
    def __init__(self):
        super().__init__()
        self._parts = []
    def handle_data(self, data):
        self._parts.append(data)
    def get_text(self) -> str:
        return ''.join(self._parts)

def render_content(content: str, url: str) -> str:
    """Return clean plain text from url for display(Markdown(...)).
    Fetches original article text via trafilatura (preferred).
    Falls back to parsing the Tavily markdown snippet via the markdown
    package to strip heading/formatting artifacts.
    Args:
        content: Tavily markdown snippet (fallback source).
        url:     Original article URL (preferred source).
    """
    if url:
        downloaded = trafilatura.fetch_url(url)
        fetched = trafilatura.extract(downloaded) if downloaded else None
        if fetched:
            return fetched.replace('\xa0', ' ')
    # Fallback: markdown → HTML → strip tags → plain text
    html = markdown.markdown(content)
    extractor = _StripHTML()
    extractor.feed(html)
    return extractor.get_text().replace('\xa0', ' ')

In [36]:
def show_tool_executions(outputs: List[Dict[str, Any]]) -> None:
    """Display tool execution results.
    Parses JSON tool payloads and calls render_content(content, url)
    to fetch clean article text for display.
    """
    for msg in outputs:
        # Response API: tool result key is "output"    -> {"type":"function_call_output","call_id":...,"output":"..."}
        # Chat Completion: tool result key was "content" -> {"role":"tool","tool_call_id":...,"content":"..."}
        payload = json.loads(msg.get("output", msg.get("content", "{}")))
        results = payload.get("results", [])
        if not results:
            note = payload.get("note", "No tool results to display.")
            display(Markdown(note))
            continue
        for result in results:
            source = result.get("title", "")
            url = result.get("url", "")
            article_content = result.get("content", "")
            published_date = result.get("published_date")
            clean = render_content(content=article_content, url=url)
            header = f"**{source}**"
            if published_date:
                header += f"\n\nPublished: {published_date}"
            display(Markdown(f"{header}\n\n{clean}"))
            display(Markdown("---"))

In [37]:
tool_calls = extract_tool_calls(response=response)
tool_outputs = execute_tool_calls(tool_calls=tool_calls)

{
    "url": "https://www.yahoo.com/news/articles/trump-calls-irans-current-leaders-034227875.html",
    "title": "Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic' - Yahoo",
    "score": 0.72489023,
    "published_date": "Mon, 30 Mar 2026 23:43:28 GMT",
    "content": "## Top Stories:. * New front in Iran war. # Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic'. Add Yahoo as a preferred source to see more of our stories on Google. Turkey's defense ministry said a ballistic missile launched from Iran entered Turkish airspace before being shot down by NATO air and missile defenses deployed in the eastern Mediterranean, the fourth such incident since the start of the war. Soon after Baghaei's remarks, Trump said in a social media post that the United States was in talks with a \"more reasonable regime\" to end the war in Iran, but he also issued a new warning over the Strait of Hormuz. His administration requested an additional 

In [38]:
show_tool_executions(outputs=tool_outputs)

**Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic' - Yahoo**

Published: Mon, 30 Mar 2026 23:43:28 GMT

Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic'
By Alexander Cornwell, Trevor Hunnicutt and Asif Shahzad
TEL AVIV/WASHINGTON/ISLAMABAD, March 30 (Reuters) - President Donald Trump warned on Monday that the U.S. would obliterate Iran's energy plants and oil wells if Tehran does not open the Strait of Hormuz, after Tehran described U.S. peace proposals as "unrealistic" and fired waves of missiles at Israel.
Israel's military said two drones from Yemen had also been intercepted on Monday, two days after the Iran-aligned Houthis entered the war by firing missiles at Israel, and that Lebanon's Hezbollah had fired rockets at Israel.
Israeli forces carried out missile strikes on what they called military infrastructure in Tehran and infrastructure used by Iran-backed Hezbollah in Beirut, leaving black smoke hanging over the Lebanese capital.
Turkey's defense ministry said a ballistic missile launched from Iran entered Turkish airspace before being shot down by NATO air and missile defenses deployed in the eastern Mediterranean, the fourth such incident since the start of the war.
Tehran remains defiant in the month-old war, which began with U.S.-Israeli attacks on Iran on February 28 and has spread across the region, killing thousands, disrupting energy supplies and hitting the global economy.
The majority of those reported killed were in Iran and Lebanon, and many were civilians. Iran has effectively blocked the Strait of Hormuz, a narrow waterway that normally carries about a fifth of global oil and liquefied natural gas supplies.
TROOPS DEPLOY AS TALKS CONTINUE
Thousands of soldiers from the U.S. Army's elite 82nd Airborne Division have started arriving in the Middle East, two U.S. officials told Reuters on Monday, part of a reinforcement that would expand Trump's options to include the deployment of forces inside Iranian territory, even as he pursues talks with Tehran.
White House press secretary Karoline Leavitt later said Trump wanted to reach a deal with Tehran before an April 6 deadline he set last week after extending an earlier deadline he had set for Iran to open the Strait of Hormuz. Leavitt said talks with Iran were progressing, adding that what Tehran says publicly differs from what it tells U.S. officials in private.
Iran said earlier on Monday it had received U.S. peace proposals via intermediaries, following talks on Sunday between the foreign ministers of Pakistan, Egypt, Saudi Arabia and Turkey.
Iranian Foreign Ministry spokesperson Esmaeil Baghaei said the proposals were "unrealistic, illogical and excessive".
More in World
"Our position is clear. We are under military aggression. Therefore, all our efforts and strength are focused on defending ourselves," he told a press conference.
Soon after Baghaei's remarks, Trump said in a social media post that the United States was in talks with a "more reasonable regime" to end the war in Iran, but he also issued a new warning over the Strait of Hormuz.
"Great progress has been made but, if for any reason a deal is not shortly reached, which it probably will be, and if the Hormuz Strait is not immediately 'Open for Business,' we will conclude our lovely 'stay' in Iran by blowing up and completely obliterating all of their Electric Generating Plants, Oil Wells and Kharg Island," Trump wrote.
Trump also threatened to attack the desalination plants that supply clean water in Iran.
The national security committee in the Iranian parliament, meanwhile, approved a bill that bans ships from the U.S., Israel and countries that unilaterally sanction Iran, from moving through the Hormuz Strait, according to state media. The bill must still be approved by the full parliament and it was not clear when or if such a vote would take place.
A Pakistani security official, whose country is trying to mediate in the war, said it appeared unlikely there would be direct U.S.-Iran talks this week.
Baghaei also said Iran's parliament was reviewing a possible exit from the Nuclear Non-Proliferation Treaty, which recognizes the right to develop, research, produce and use nuclear energy as long as nuclear weapons are not pursued.
Trump has cited the prevention of Iran from obtaining nuclear weapons as a reason for attacking the country on February 28. Tehran denies it is seeking a nuclear arsenal.
Israeli Prime Minister Benjamin Netanyahu, in an interview with U.S. media outlet Newsmax, declined to give a timeline for achieving his country’s objectives in the war. While he said that "it's definitely beyond the halfway point," he later clarified that he meant in terms of missions, not time.
OIL MARKETS BRACE FOR TURMOIL
The White House said Trump was considering asking Arab nations to pay for the cost of the war. "It's an idea that I know that he has and something that I think you'll hear more from him on," Leavitt said in response to a reporter's question about the idea.
His administration requested an additional $200 billion in funding for the war, which faces stiff opposition in the U.S. Congress, which must approve new spending.
Iran has fired on Arab Gulf states during the conflict and war has been reignited between Israel and Hezbollah in Lebanon. Three members of the U.N. peacekeeping mission in Lebanon (UNIFIL) were killed in two separate incidents in southern Lebanon after a bloody weekend in which Lebanese journalists and medics were killed in Israeli strikes.
Benchmark oil prices extended gains on Monday, with Brent crude futures on course for a record monthly rise.
The Houthis' attacks on Israel raised the prospect that they could target and block a second important shipping route, the Bab el-Mandeb Strait.
The oil market has all but discounted the prospect of a negotiated end to the war and "is bracing for a sharp escalation in military hostilities," said Vandana Hari of oil-market provider Vanda Insights.
The International Monetary Fund warned that war in the Middle East has caused serious disruption to the economies of frontline countries, and is dimming the outlook for many economies that had just started to recover from previous crises.
G7 finance leaders also said they were ready to take "all necessary measures" to safeguard energy market stability and limit broader economic spillovers from recent volatility.
(Reporting by Reuters bureaux; Writing by Stephen Coates, Timothy Heritage, Keith Weir, Simon Lewis and Brad Brooks; Editing by Gareth Jones and Matthew Lewis)

---

**Iran strikes fully loaded oil tanker off Dubai coast as gas reaches $4 per gallon in US. Follow live updates. - The Boston Globe**

Published: Tue, 31 Mar 2026 16:48:47 GMT

President Trump will speak at 9 p.m. on Wednesday about Iran, and he told reporters Tuesday that the responsibility for keeping the Strait of Hormuz open should belong with countries that rely on it, rather than the US. Trump expressed frustration earlier Tuesday with allies who have been unwilling to do more to support the US war effort, telling them to “go get your own oil.”
Meanwhile, US stocks surged to their best day since last spring, and the Dow Jones Industrial Average soared 1,125 points on Tuesday as doubt swung back to hope on Wall Street about a possible end to the war with Iran.
See a recap.
Iranian drone attack starts fire at Kuwait International Airport — 10:45 p.m.
By the Associated Press
A drone attack by Iran and its allies hit a fuel tank at Kuwait International Airport, sparking a fire, authorities said.The state-run KUNA news agency said the attack early Wednesday sparked a “large fire” at the airport.
It said there were no immediate injuries from the attack and firefighters were working to control the blaze.
Kuwait International Airport has been attacked before by Iran during the war. The KUNA report suggested the attack may have been launched by Iranian-supported militias in Iraq with Tehran’s support.
In another strike, Bahrain said early on Wednesday morning that it was working to extinguish a fire at a business facility that resulted from an Iranian attack.
Major airlines in United Arab Emirates say country is barring Iranian travelers — 10:12 p.m.
By the Associated Press
The United Arab Emirates has barred Iranians from entering or transiting the country as the war rages, three major airlines said Wednesday.
Long-haul carriers Emirates and Etihad, as well as the lower-cost airline FlyDubai, made the announcements on their websites.
Entry rules can sometimes be opaque in the autocratic United Arab Emirates, a federation of seven sheikhdoms, the three airlines agreed on the order. It said holders of 10-year Golden Visa residency permits could still enter the country.
Authorities have offered no official comment. But Dubai has already shut down the city-state’s Iranian Hospital and Iranian Club, institutions that date back to the time of the shah.
Tanker is attacked off Qatar coast — 9:54 p.m.
by the Associated Press
A tanker came under attack off the coast of Qatar early Wednesday, authorities said.
The British military’s United Kingdom Maritime Trade Operations center announced the attack happened, saying a projectile slammed into the side of the ship.
It said no environmental damage was done and the tanker’s crew was safe.
On Tuesday, a fully loaded Kuwaiti oil tanker came under attack off Dubai. Over 20 ships across the Persian Gulf have been attacked by Iran since the war began.
Saudi Arabia says it intercepted 2 drones — 8:35 p.m.
By the Associated Press
Early on Wednesday morning, Saudi Arabia said it had intercepted and destroyed two drones in the previous hours.
A Gulf ally of the United States, the country has been targeted by Iran repeatedly. This week, more than two dozen American service members were injured from missile and drone attacks on a Saudi air base.
White House says Trump will deliver prime-time address Wednesday evening to update public on Iran war — 8:02 p.m.
By the Associated Press
White House press secretary Karoline Leavitt posted on X that Trump will “give an Address to the Nation to provide an important update on Iran” at 9 p.m. EDT on Wednesday.
Her post came shortly after Trump told reporters in the Oval Office on Tuesday evening that U.S. forces could leave Iran in “two or three weeks.”
TUNE IN: Tomorrow night at 9PM ET, President Trump will give an Address to the Nation to provide an important update on Iran.
— Karoline Leavitt (@PressSec) March 31, 2026
Trump says the US is ‘finishing the job’ in Iran — 7:49 p.m.
By the Associated Press
President Trump predicted the Us will be done with the conflict “within maybe two weeks, maybe a couple of days longer to do the job. But we want to knock out every single thing they have.”
Despite repeatedly listing four or five objectives for the war, Trump said he “had one goal: They will have no nuclear weapon and that goal has been attained.” He did not explain how he felt that goal had been reached.
He said the U.S. may strike a deal with Iran before the next few weeks are up but said, if not, “We’ll hit some bridges, got a couple of nice bridges in mind. But if they come to the table, that’ll be good.”
348 US troops have been wounded in the Iran war — 7:26 p.m.
By the Associated Press
The formal injury count, provided by Capt. Tim Hawkins, spokesman for U.S. Central Command, says six service members were wounded seriously.
Central Command said last Friday in a previous update that 303 troops were wounded, 10 of them seriously.It was not clear why the tally of serious injuries has since been decreased.
Hawkins says of the total wounded to date, 315 service members have returned to duty.
It’s not clear whether the official U.S. military count includes the injured from multiple attacks on Saudi Arabia’s Prince Sultan air base last week.
Since the Iran war began, 13 U.S. service members have been killed in combat.
2 people were killed, several more were injured in strikes in Beirut — 7:07 p.m.
By the Associated Press
Two people were killed and three were injured very early on Wednesday when a strike hit a car in the Khaldeh area south of the capital, Lebanon’s health ministry said.
More were injured as strikes hit another area of the city, according to a local hospital.
Syria will stay out of the regional war, interim president says — 6:38 p.m.
By the Associated Press
Syrian interim President Ahmad al-Sharaa was speaking at an event hosted by the Chatham House think tank during a visit to London Tuesday.
“Certainly, unless it is directly targeted by any party, Syria will stay out of this conflict,” he said in response to a question about the ongoing Iran war. “Fourteen years of war are enough for Syria, during which we have paid a very high price. We are not ready to go through a new experience. Anyone who has been through war knows the value of peace.”
Syria, which is trying to recover from its own lengthy civil war, is one of the few countries in the region that has managed to remain on the sidelines of the war.
Trump says US ‘will not have anything to do with’ what happens in the Strait of Hormuz — 6:18 p.m.
By the Associated Press
The president told reporters that the responsibility for keeping the vital waterway open would instead belong with countries that rely on it.
He said there’s “no reason for us to do this.”
“That’s not for us. That’ll be for France. That’ll be for whoever’s using the strait,” Trump said.
His comments came after he lashed out earlier Tuesday at U.S. allies for not doing more to support the U.S.’s efforts in the Iran war.
US officials raise concerns about indirect talks with Iran, Islamabad as meeting place — 5:18 p.m.
By the Associated Press
While welcoming Pakistan’s desire to help mediate in the Iran conflict, U.S. officials have nonetheless soured on the idea of using third-party mediators to do anything more than initiate talks.
These officials, who spoke on condition of anonymity due to the sensitivity of the diplomatic effort, said the experience with Oman as mediator before the war had clearly not worked and that direct negotiations, if they can be started, are far more efficient.
The officials also expressed doubts about Islamabad being chosen as a venue for any upcoming negotiations, mainly because of security risks. Traveling there would mean flying over one of two war zones — Russia-Ukraine or the various Middle East conflicts — or taking a very long route over Asia.
A U.N. diplomat familiar with the talks, who also spoke on the condition of anonymity to discuss private negotiations, acknowledged the risk given Pakistan’s proximity to Iran, but added that given that the South Asian country is one of few in the region with no U.S. airbases, it’s a safer bet.
US warns Americans in Saudi Arabia to shelter in place — 5:03 p.m.
By the Associated Press
The U.S. State Department’s consular affairs branch said that it was “tracking reports” that hotels and other gathering points, such as U.S. business and educational institutions, could be targeted.
The U.S. Embassy in Riyadh also told U.S. government employees to shelter in place.
Iran has repeatedly threatened educational and business places connected to Americans in the region during the war.
Israel now blames Hezbollah for all 3 UN peacekeeper deaths — 4:56 p.m.
By the Associated Press
UN Ambassador Danny Danon said Israel now blames Hezbollah for all three of the recent peacekeeper deaths in southern Lebanon, citing explosive devices Monday near Bani Ayan and the shelling of a UN peacekeeper position Saturday.
He provided no evidence, and a UN spokesman said the investigation is ongoing. The three Indonesian peacekeepers were killed as Israel and Hezbollah have been engaged in fighting.
Late on Tuesday, Israel’s military said its troops were not present in the southern Lebanon area where an explosion killed two UNIFIL peacekeepers on Monday. It also said no explosive device had been placed in the area by Israeli soldiers.
A third US aircraft carrier is slated to join the Iran war — 4:54 p.m.
By the Associated Press
The aircraft carrier USS George H. W. Bush deployed Tuesday and is slated to head to the Middle East, two US officials said.
The move would make the Bush the third aircraft carrier to have supported the Iran war, along with the USS Gerald R. Ford, which is now undergoing repairs, and USS Abraham Lincoln, which arrived in the region in January.
The Ford now is docked in Croatia after undergoing repairs in a nearby naval base in Crete following a fire in a laundry room on March 12.
The officials, who spoke on condition of anonymity to discuss sensitive military plans, noted that plans for ship deployments are always subject to change.
The Bush is deploying from Norfolk, Virginia, with three destroyers: USS Ross, USS Donald Cook, and USS Mason.
Revolutionary Guard threatens US tech firms, saying they help spy on Iran — 4:11 p.m.
By the Associated Press
Iran’s Revolutionary Guard has threatened to attack US companies including some of the biggest tech giants after accusing them of being involved in “terrorist espionage” operations in Iran.
A statement carried by Iranian state media listed 18 companies whose offices in the Middle East region they claimed would be legitimate targets for Iran starting Wednesday. Those include Apple, Microsoft, Google and others.
The Guard has launched missiles and drones at Israel and Gulf Arab countries since the United States and Israel began bombing Iran on Feb. 28.
Earlier this month, Iranian drone strikes caused damage to three Amazon Web Services facilities in the United Arab Emirates and Bahrain.
Canada condemns Israel’s invasion of Lebanon as ‘illegal’ and calls for a cease-fire — 4:06 p.m.
By the Associated Press
Canadian Prime Minister Mark Carney said his nation “condemns” the invasion as a violation of Lebanon’s’ s sovereignty.
As Israeli ground forces push deeper into the country, Carney said Lebanon’s government shares Israel’s stated goal of curbing Hezbollah, even as Israel cites the presence of fighters to justify the offensive.
Carney made the call for the ceasefire Tuesday while speaking to reporters in Quebec.
US stocks surge to their best day since last spring — 4:05 p.m.
By the Associated Press
The Dow Jones Industrial Average soared more than 1,100 points as doubt swings back to hope on Wall Street for a possible end to the war with Iran.
The S&P 500 leaped 2.9 percent Tuesday for its largest gain since May.
Just a day before, worries about the war had sent the main measure of Wall Street’s health more than 9 percent below its all-time high set early this year.
The Dow Jones Industrial Average jumped 1,125 points and the Nasdaq composite surged 3.8 percent.
Oil prices eased to fuel the rally.
Netanyahu says Iran can no longer threaten Israel’s existence — 3:03 p.m.
By the Associated Press
Netanyahu says Israel has scored major achievements in weakening the Iranian regime and its military capabilities.
He said Israeli forces are systematically crushing the Iranian regime even if it still has weapons launch capabilities.
The Israeli prime minister adds that Israel hopes to soon be able to speak about new alliances in the region.
Journalist kidnapped in Iraq identified — 3:00 p.m.
By the Associated Press
AI-Monitor, a regional news site covering the Middle East, has identified the journalist kidnapped Tuesday in Baghdad as Shelly Kittleson, a freelancer who contributed to the publication. In a statement, Al-Monitor said it is “deeply alarmed” by her kidnapping.
“We call for her safe and immediate release,” the statement said. “We stand by her vital reporting from the region and call for her swift return to continue her important work.”
Kittleson has been a longtime freelancer in the region, reporting extensively from Syria and Iraq.
Iraqi officials had earlier said that a foreign journalist was kidnapped and that security forces were pursuing the captors.
The US State Department said in a statement that it is tracking the reports and that the “Trump Administration has no higher priority than the safety and security of Americans.”
Russia’s Putin speaks by phone with UAE president — 1:41 p.m.
By the Associated Press
Russian President Vladimir Putin expressed “serious concern” about the ongoing war to his Emirati counterpart, Mohamed bin Zayed Al Nahyan, during their phone conversation Tuesday.
The Kremlin readout of the call said they emphasized “the importance of a swift cessation of hostilities and intensification of political and diplomatic efforts towards a peaceful resolution of the conflict, while respecting the legitimate interests of all states in the region, with which Russia traditionally maintains friendly and mutually beneficial relations.”
Israel gives more information on 4 soldiers killed in Lebanon — 1:13 p.m.
By the Associated Press
In an update Tuesday evening, the Israeli military said soldiers from the Nahal reconnaissance unit engaged in close combat after militants opened fire on them overnight in southern Lebanon. With support from aircraft and tanks, the militants were killed, according to the statement.
The IDF first announced the four soldiers’ deaths at around 6:22 a.m. local time. Additionally, one soldier was severely wounded and another moderately injured.
The journalist kidnapped in Iraq is a woman with U.S. citizenship, officials say — 1:08 p.m.
By the Associated Press
Two Iraqi security officials said the kidnapped journalist was a woman with U.S. citizenship and that two cars were involved in the kidnapping, one of which crashed and was apprehended, while the car carrying the journalist fled the scene.
A foreign journalist has been kidnapped in Iraq, the country’s interior ministry says — 12:55 p.m.
By the Associated Press
The ministry did not identify the journalist or give further details on their nationality.
In a statement Tuesday, it said that security forces had launched an operation to track down the kidnappers, “acting on precise intelligence and through intensive field operations” after intercepting a vehicle belonging to the kidnappers that overturned as they tried to flee.
One suspect was arrested and one of the vehicles used in the kidnapping was seized, but others remain on the loose, the statement said. A spokesperson for the U.S. Embassy in Baghdad declined to comment.
Iran remains a stubborn foe after absorbing massive US-Israeli attacks — 12:48 p.m.
By the Associated Press
Iran’s missiles continue to penetrate Israeli airspace and kill civilians. Its cheap drones slip through its neighbors’ air defenses, shattering Gulf Arab nations’ carefully curated images of invincibility and wounding U.S. troops. Its threats to attack oil and gas tankers strangle the Strait of Hormuz, sending energy prices soaring.
To maintain its leverage, Iran just needs to withstand the conflict long enough to pressure Washington to seek an off-ramp, experts say.
“Their strategy is to try to cause sustained pain and to drive up the costs of the war for the U.S.,” said Kelly Grieco, an expert in U.S. military strategy and operations who is a senior fellow at the Washington-based Stimson Center think tank.
Iran is launching fewer missiles than at start of the war, but is deploying more low-flying drones that are harder to intercept.
France urges action over Israeli threats and attacks on UN peacekeepers — 12:27 p.m.
By the Associated Press
France’s UN Ambassador Jerome Bonnafont told an emergency Security Council meeting that members must take action and not merely settle for condemning Israel’s “grave” threats and attacks on peacekeepers, including French forces
“It must act so that these kinds of acts do not happen again,” he said, calling on Israel to provide immediate guarantees that deconfliction procedures exist and will be respected.
Bonnafont, who requested the meeting after three Indonesian peacekeepers were killed, also urged Israel and Hezbollah to respect the safety of UN forces, warning that those who do not “should be accountable before this council and the entire international community.”
He added that “Hezbollah must stop taking the Lebanese people hostage” in a war he said is driven by Iran, and said France is working toward a diplomatic solution.
Iran’s imprisoned Nobel Peace Prize laureate Narges Mohammadi may have suffered a heart attack — 12:15 p.m.
By the Associated Press
Mohammadi’s legal team, accompanied by one family member, visited her in Zanjan Prison on March 29, according to a statement from the Free Narges Coalition campaign.
“Her general health was extremely poor, and she appeared pale and weak with significant weight loss,” it said in a statement, then cited her fellow inmates as saying she was found unconscious in her bed with her eyes rolled back on March 24.
“Despite this medical emergency, and evident indications of a heart attack, authorities refused to transfer Mohammadi to a hospital or allow her to visit a specialist,” the statement said.
Mohammadi has a heart condition and suffered multiple heart attacks while imprisoned before undergoing emergency surgery in 2022, her supporters say.
US ambassador says the world should ‘reserve judgment’ on peacekeeper killings during investigation — 12:10 p.m.
By the Associated Press
US Ambassador Mike Waltz called for a “pause” during an emergency Security Council meeting on the killings, while the United Nations figures out whether Israel or Hezbollah militants are to blame.
He said the UN must “fully investigate and assess the circumstances of these tragic incidents,” even as countries share in the grief.
Waltz also called for changes to UN peacekeeping operations, saying the council owes troops not just condolences but “a wise approach” that recognizes “terrorists have no respect for the norms of international law.”
UK defense secretary defends ties with US despite criticism from Hegseth — 11:45 a.m.
By the Associated Press
British Defense Secretary John Healey said Tuesday that the US remains a key ally despite criticism from his American counterpart, Pete Hegseth, that the UK had not deployed its navy to the Middle East.
“The U.S. is a uniquely close ally to the U.K.,” Healey said in Qatar. “We do things as two nations that no other militaries or intelligence services do. And my job as defense secretary is to make sure that we can, in this Middle Eastern conflict, defend Britain and British people, and we are; and British bases, and we are; and British allies and partners, and we are.”
Hegseth sniped at the UK for not sending warships to the region, saying at a Washington news conference that “last time I checked there was supposed to be a big, bad Royal Navy that could be prepared to do things like that as well.”
Healey announced that the UK is sending more missile and air defense systems to Bahrain, Kuwait and Saudi Arabia, and extending the use of its Typhoon fighter jets in Qatar.
World Health Organization says US-Israeli strikes have hit near its Tehran offices — 11:43 a.m.
By the Associated Press
In a social media post, WHO’s director general said the windows in its offices in Iran’s capital were shattered after strikes in the last two days, but that no one was injured.
“Strikes impacting the operations and damaging the premises of WHO and other @UN agencies, the locations of which have been clearly identified, cannot be tolerated and must be avoided at all costs,” Tedros Adhanom Ghebreyesus said.
Anti-government protester calls Iran’s territory a ‘red line’ — 11:41 a.m.
By the Associated Press
A young anti-government activist in Iran says he would volunteer with the army if the United States launches ground operations, calling the country’s territorial integrity a “red line.”
“If the idea of occupying islands or part of my country’s territory is implemented, I will definitely be available as a soldier to defend the Iranian nation,” said the 25-year-old from the northern town of Babol, speaking on condition of anonymity because of fears for his safety.
The activist, who joined protests before the war, said he received a weekend text urging volunteers to join national “defense” efforts. He said he would not serve with the Revolutionary Guard, which has crushed past unrest, but would join the regular army.
Iran has seen multiple text-message campaigns urging enlistment, although it is unclear how many recruits they have generated. The country also requires military service for most men over 18, with limited exemptions.
Israeli envoy blames Hezbollah for killing 2 UN peacekeepers but offers no details — 11:17 a.m.
By the Associated Press
U.N. Ambassador Danny Danon said Israel “can confirm now that UNIFIL forces were hit by Hezbollah explosive devices in an incident near Bani Ayan in southern Lebanon.”
He offered condolences to their families but provided no details about the circumstances of their deaths Monday. A third UNIFIL peacekeeper was killed Sunday.
Speaking to reporters Tuesday, Danon accused Hezbollah of launching attacks from civilian buildings and infrastructure near U.N. positions. He said the Iran-backed militant group continues to operate freely, in violation of a U.N. Security Council resolution calling for its disarmament and the deployment of Lebanese forces across the country. The Lebanese government “has done neither,” Danon said.
Lebanon has issued condolences over the three Indonesian peacekeepers’ deaths, but neither the government nor Hezbollah have addressed allegations that the militant group was responsible.
FIFA’s president attends soccer game between Iran and Costa Rica — 11:09 a.m.
By the Associated Press
At an emergency U.N. Security Council session Tuesday, world powers denounced the two incidents in the last two days that led to the killing of three peacekeepers in southern Lebanon, saying it’s part of a pattern of aggression towards the officers carrying out the mission.
“These are sadly not the only dangerous incidents faced by UNIFIL’s courageous peacekeepers,” Jean-Pierre Lacroix, the U.N. peacekeeping chief, said during his briefing. “There has been a worrying increase in denials of freedom of movement and aggressive behavior.”
He described several incidents in the last week where the Israel Defense Forces fired warning shots at a UNIFIL patrol and days later another patrol was subjected to heavy small arms fire “from a group of approximately 20 individuals blocking the road.”
Lacroix added that the investigation into the origin of the attacks is ongoing and it’s not clear which side was responsible for the death of the three Indonesian officers.
Earlier Tuesday, Italy and France expressed concern over the attacks against U.N. personnel and Turkey has condemned such attacks.
FIFA’s president attends soccer game between Iran and Costa Rica — 10:50 a.m.
By the Associated Press
Players on Iran’s national soccer team used the friendly international game against Costa Rica on Tuesday to honor children allegedly killed by U.S. and Israeli airstrikes and bombardment of their country.
There were no spectators at the stadium in Antalya in southern Turkey but FIFA’s President Gianni Infantino was present at the game.
They were joined by Iran’s coach Amir Ghalenoei, Iran’s Football Federation Vice President Mehdi Mohammad Nabi and staff members, posing with the photographs of children in their hands while singing the national anthem ahead of the match that served Iran as a World Cup warmup ahead of the tournament being co-hosted by the United States, Mexico and Canada.
China and Pakistan to promote five points aimed at ending the war — 10:27 a.m.
By the Associated Press
China and Pakistan agreed to promote a five-point proposal aimed at restoring peace in the Middle East after a monthlong war.
Chinese Foreign Minister Wang Yi on Tuesday received his Pakistani counterpart, Ishaq Dar, and both agreed on the five points they’ll pursue: an immediate cessation of hostilities, the start of peace talks as soon as possible, ensuring the safety of non-military targets, guaranteeing the safety of navigation through the Strait of Hormuz, and safeguarding the primacy of the U.N. charter.
Chinese state media and Pakistan’s Foreign Ministry announced the agreement.
Both countries called on all parties to follow the proposals, but they didn’t mention any other concrete steps.
Dar traveled to Beijing as Pakistan has been acting as a mediator between Iran and the United States. The South Asian country is using its relatively good ties with both Washington and Tehran to try to help end the war.
Seizing Kharg Island would risk US troops’ lives and may not end Iran war, experts say — 10:01 a.m.
By the Associated Press
President Trump is threatening to deploy ground troops to seize critical oil infrastructure on Iran’s Kharg Island, a military gambit experts say would risk American lives and could still fail to end the war.
If Trump wants to hobble Iran’s oil industry for leverage in negotiations, a better option might be setting up a blockade at sea against ships that have filled up at Kharg Island’s oil terminals, the experts said.
The island — located on the other side of the Persian Gulf from U.S. bases in Kuwait and Saudi Arabia — is the beating heart of Iran’s oil industry, through which 90% of its exports pass.
“Putting people on the ground might be the most psychologically compelling way of striking a blow at Iran,” said Michael Eisenstadt, a former U.S. military analyst who now directs the Military and Security Studies Program at the Washington Institute for Near East Policy.
“On the other hand, you’re putting your own troops at jeopardy,” said Eisenstadt, a retired Army reserve officer who served in Iraq. “It’s not far from the mainland. So they can potentially rain a lot of destruction on the island, if they’re willing to inflict damage on their own infrastructure.”
US stocks bounce back as crude oil prices stabilize — 9:57 a.m.
By the Associated Press
U.S. stocks are bouncing back as the spike for oil prices caused by the war with Iran slows.
The S&P 500 climbed 1.2% early Tuesday. A day earlier, it closed more than 9% below the all-time high it set early this year. The Dow Jones Industrial Average rose 410 points, and the Nasdaq composite added 1.6%.
Steadying oil prices took some pressure off Wall Street. The worry is that if oil prices stay high for a long time because of the war, it could set off a brutal blast of global inflation. Treasury yields ease again in the bond market.
Death toll in Lebanon reaches 1,268 since the latest Israel-Hezbollah war began — 9:48 a.m.
By the Associated Press
The Health Ministry in Beirut said Tuesday that 21 people were killed and 70 wounded over the past 24 hours.
The ministry says 1,268 have been killed and 3,750 wounded since the latest Israel-Hezbollah war began March 2.
The dead include 125 children and 88 women, the ministry says.
Israeli airstrike kills a father and his 5-year-old son in southern Gaza — 9:46 a.m.
By the Associated Press
That’s according to health officials at Nasser hospital, where the casualties were taken Tuesday.
The Israeli military did not immediately respond to a request for comment.
In the courtyard of Nasser hospital, the family and acquaintances of the father and his son gathered for the funeral prayer, carrying their bodies in white burial shrouds, in tears and agony.
The Gaza Strip has seen near-daily Israeli fire and strikes since a fragile ceasefire was reached in October and nearly 709 Palestinians have been killed since, according to figures from the Hamas-run Gaza Health Ministry. Israel and Hamas have traded accusations of violating the ceasefire.
Gaza’s militants have sat out the current Iran conflict.
Iranian foreign minister warns against targeting infrastructure — 9:35 a.m.
By the Associated Press
Iranian Foreign Minister Abbas Araghchi on Tuesday issued a warning against Israel “unashamedly” bombing pharmaceutical companies as part of the Iranian infrastructure the U.S. and Israel have been targeting since the war began.
“Their intentions are clear. What they’ve gotten wrong is that they’re not dealing with defenseless Palestinian civilians. Our Powerful Armed Forces will severely punish aggressors,” he wrote on X.
Israel’s defense minister outlined plans for Israeli invasion of Lebanon — 9:33 a.m.
By the Associated Press
Speaking to military officials, minister Israel Katz reiterated that the military aims to control the area south of the Litani River — some 20 miles (about 30 kilometers) north of the border.
He said Israel will prohibit the return of 600,000 Lebanese who fled the area over the last few weeks until safety and security were “ensured” for residents of Israel’s north.
Hezbollah and Israel have exchanged consistent cross-border fire since the latest flareup that began March 2. He said all homes in the Lebanese villages directly across the border from Israel would be demolished “in order to remove once and for all the threats near the border from residents of the north.”
Indonesian government implements efficiency measures as the Iran war squeezes the energy sector — 9:30 a.m.
By the Associated Press
The Indonesian government has started to implement a work-from-home policy for civil servants as an adaptive and proactive measure in response to global developments of the ongoing war in the Middle East that’s straining global supply chains, particularly in the energy sector.
“Implementing work-from-home arrangements for civil servants at the central and regional levels, with one workday per week on Fridays,” Coordinating Economic Minister Airlangga Hartarto said in a broadcasted conference Tuesday.
The government is also implementing mobility efficiency measures that include a 50% reduction in official vehicle use, except for operational purposes and electric vehicles, and encouraging the use of public transportation.
The measure include a reduction of up to 50% in domestic business trips and up to 70% in international business trips, Hartarto said.
Recommendations regarding working from home and efficiency have also been provided to the private sector, taking into account the needs and characteristics of each business.
Trump says the US isn’t pulling assets from around the Strait of Hormuz just yet — 9:15 a.m.
By the Associated Press
“At some point I will, not quite yet, but countries have to come in and take care of it,” he told CBS News in a telephone interview Tuesday. “Iran has been decimated, but they’re going to have to come in and do their own work.”
The conversation followed Trump’s social media post in which he lashed out at allies over their unwillingness to help the U.S. reopen the critical passageway. He said Iran has been “decimated” and no longer poses a “real threat.”
“Let them come up and take it. They didn’t want to give a hand to anybody. NATO is terrible, and they’re all terrible,” Trump said. “So if they want oil, come up and grab it.”
Hegseth says Britain and other allies should ‘step up’ to open the Strait of Hormuz — 9:05 a.m.
By the Associated Press
“There are countries around the world who ought be prepared to step up on this critical waterway as well,” Hegseth said Tuesday, speaking at a news conference from the Pentagon. “It’s not just the United States Navy. Last time I checked, there was supposed to be a big, bad Royal Navy that could be prepared to do things like that as well.”
In a social media post Tuesday, President Trump said nations upset by high fuel prices should “go get your own oil” as as average U.S. gas prices shot past $4 a gallon.
If countries don’t stand with you, it’s not much of an alliance, Hegseth says — 9:02 a.m.
By the Associated Press
Defense Secretary Pete Hegseth said Tuesday the U.S. undertook the war in Iran for the “free world” and questioned the value of the NATO alliance if those countries don’t stand with America.
Hegseth and Chairman of the Joint Chiefs of Staff Gen. Dan Caine have wrapped up a news conference at the Pentagon. Hegseth pointed to a social media post from President Donald Trump about allies and said Iranian missiles don’t reach the U.S. but could hit allies and others.
“The president’s pointing out you don’t have much of an, an alliance if you have countries that are not willing to stand with you when you need them,” Hegseth said.
Hegseth won’t say if US will put ‘boots on the ground’ in Iran — 8:59 a.m.
By the Associated Press
Defense Secretary Pete Hegseth declined to tell reporters Tuesday whether or not the U.S. military will deploy ground troops against Iran.
“You can’t fight and win a war if you tell your adversary what you are willing to do or what you are not willing to do to include boots on the ground,” he said.
Hegseth added: “Our adversary right now thinks there are 15 different ways we could come at them with boots on the ground. And guess what? There are. “
Hegseth also said talks with Iran to end the conflict are ongoing.
“We don’t want to have to do more militarily than we have to,” he said. “But I didn’t mean it flippantly when I said, in the meantime, we’ll negotiate with bombs.”
General Caine says US strikes are focused on Iranian naval targets and defense industrial sites — 8:52 a.m.
By the Associated Press
Speaking at a news conference from the Pentagon on Tuesday, Gen. Dan Caine, chairman of the Joint Chiefs of Staff, said U.S. military action in Iran remains focused on “targeting their minelaying capability, their naval assets.”
“We’ve taken out again more than 150 ships,” Caine said, adding that attack helicopters are now joining in the effort targeting Iranian naval targets.
Another key objective of the war is disabling Iran’s defense industrial base, including nuclear research sites, Caine said.
“This includes factories, warehouses, nuclear weapons research and development labs, and the associated infrastructure required for Iran to reconstitute its combat capability,” Caine said.
UN’s refugee agency says more than 200,000 people crossed from Lebanon into Syria in March — 8:46 a.m.
By the Associated Press
The agency says the tally follows renewed fighting between Israel and Hezbollah in the war and covers a period from March 2 to March 27.
The vast majority – nearly 180,000 – were Syrians returning to their war-battered country, in addition to more than 28,000 Lebanese.
“Most are people fleeing the intense Israeli bombardments. They arrive exhausted, traumatized and with very, very few belongings,” UNHCR’s representative in Syria, Aseer Al-Madaien, told a U.N. briefing in Geneva by video from Damascus.
The agency has already helped more than 3 million people displaced both within Syria and abroad who’ve returned home following the fall of President Bashar Assad in December 2024.
Unlike the 2024 Israel-Hezbollah war, when Lebanese could flee across the border without visas, the current Syrian government has restricted the entry of Lebanese unless they have residency in Syria, a Syrian spouse or parent, or other exceptional circumstances.
Hegseth says he visited US troops fighting in Iran war — 8:20 a.m.
By the Associated Press
Speaking at a news conference from the Pentagon on Tuesday, Defense Secretary Pete Hegseth said he visited American service members in the Middle East. He said he wouldn’t disclose the base names or locations for operational security.
Hegseth said he visited areas under the responsibility of U.S. Central Command on Saturday for about half a day.
“Suffice it to say, the trip was in honor,” Hegseth told reporters. “I had a chance to bear witness, and I witnessed the best of America.”
Italy says its relationship with the US is ‘solid’ after reports it denied use of a base to the US — 8:03 a.m.
By the Associated Press
The Italian government says its relationship with the U.S. is “solid and based on full and loyal cooperation,” following reports it denied the use of a Sicilian base to U.S. aircraft headed to the Middle East.
The government of Prime Minister Giorgia Meloni said in a statement that Italy is acting “in full compliance with existing international agreements and the government’s guidelines expressed in parliament.”
It said each request for military use of Italian bases is examined on a case-by-case basis, its longstanding procedure.
“No critical issues or frictions with international partners have been registered,” it added.
Israel begins new wave of airstrikes on Hezbollah in Beirut — 7:58 a.m.
By the Associated Press
Israel’s military says it has begun a new wave of airstrikes on Hezbollah infrastructure in Beirut.
New airstrikes target Tehran — 7:51 a.m.
By the Associated Press
Airstrikes hit Iran’s capital, Tehran, on Tuesday afternoon as air defenses could be heard firing.
Trump says nations upset by high fuel prices should ‘go get your own oil’ — 7:37 a.m.
By the Associated Press
President Trump says nations upset by high fuel prices should “go get your own oil” as Iran maintains its chokehold on the Strait of Hormuz.
His comments in a social media post on Tuesday came as average U.S. gas prices shot past $4 a gallon.
Gasoline and diesel prices jump in UAE — 7:22 a.m.
By the Associated Press
The United Arab Emirates set sharply highly gasoline and diesel fuel prices on Tuesday for the coming month, with gasoline going up by over 30% and diesel jumping up more than 70%.
The UAE government sets the price monthly in line with international pricing, which has spiked over the Iran war and Tehran maintaining its chokehold over the Strait of Hormuz.
In the UAE, diesel fuel will jump to 4.69 dirhams ($1.28) a liter, up from 2.72 dirhams (74 cents). The new price is $4.38 a gallon for diesel, lower than the average gallon of diesel in the U.S., which sits at $5.45 a gallon.
Premium gasoline in the UAE will be 3.39 dirhams (92 cents) a liter. That’s $3.49 a gallon, where premium on average in the U.S. is $4.90 a gallon.
Egyptian president briefs Russia leader in call — 7:19 a.m.
By the Associated Press
Egyptian President Abdel-Fattah el-Sissi briefed Russian leader Vladimir Putin about Egypt’s efforts to de-escalate in the region during a phone call Tuesday, according to el-Sissi’s office.
He said Russia is able to help put an end to the war, a statement from the office said.
Italy refuses US permission to use air base — 7:11 a.m.
By the Associated Press
Italy has refused permission for U.S. military assets to use the Sigonella air base in Sicily for an operation linked to the Middle East offensive, an official said.
The refusal was issued a few days ago and concerned U.S. aircraft including bombers, which were intended to land at the base before continuing toward the Middle East, the official said on condition of anonymity because they were not authorized to speak publicly.
Under agreements governing U.S. military use of bases in Italy, Rome must be formally consulted and grant approval before operations can proceed.
The request was denied because Italian authorities were not alerted in time and the U.S. assets included bombers, the official said.
Prime Minister Giorgia Meloni’s government has pledged decisions involving military actions would require parliamentary approval.
Italy’s defense ministry did not immediately issue a statement on the decision.
Turkey condemns attacks on UN peacekeepers in Lebanon — 6:52 a.m.
By the Associated Press
Turkey has denounced attacks targeting personnel with the United Nations Interim Force in Lebanon as a serious violation of international law.
The statement from the Turkish Foreign Ministry issued Tuesday added that those responsible for attacking UNIFIL peacekeepers must face justice.
The statement criticized Israel’s invasion of Lebanon, saying it was deepening regional instability, and issued a call to the international community to end “Israel’s expansionism, aggression, and occupation.”
Korean Air to enter cost-cutting mode — 6:16 a.m.
By the Associated Press
Korean Air says it is entering an “emergency management mode” to cope with soaring fuel costs triggered by the war in the Middle East.
South Korea’s biggest airline said Tuesday it is setting internal targets to reduce costs that are not essential to flight operations.
The company said cost-cutting measures would be implemented in phases starting in April, but didn’t specify what they would be or whether they would include major flight reductions.
It added that fuel costs for April are expected to be more than double its previous projections stated in annual business plans.
Satellite images show damage to Qatar’s Al Udeid Air Base — 6:00 a.m.
By the Associated Press
Satellite photos analyzed by The Associated Press show damage after an Iranian attack targeting Qatar’s Al Udeid Air Base.
The March 15 photo from Planet Labs PBC shows damage to one of the massive air base’s buildings.
Qatar and the U.S. have not acknowledged the damage.
Al Udeid serves as the forward headquarters of the U.S. military’s Central Command, which is prosecuting the war.
Information has so far been scarce about the damage being done across the Middle East, particularly inside closed military facilities, since the war started Feb. 28.
The images come from Planet Labs PBC, a San Francisco-based firm used by media outlets, including the AP.
Planet Labs has put a two-week delay on its imagery becoming public, citing concerns its imagery could be used by “adversarial actors.”
Egypt minister discusses mediation with Arab counterparts — 5:50 a.m.
By the Associated Press
Egypt Foreign Minister Badr Abdelattay briefed foreign ministers from Saudi Arabia, the United Arab Emirates, Qatar and Jordan about the latest round of mediation efforts.
Abdelattay and Pakistani and Turkish counterparts met over the weekend in Islamabad for talks aimed at bringing Iran and the United States back to the negotiating table, according to Egypt’s Foreign Ministry.
Abdelattay discussed the meeting’s outcome and ongoing efforts to stop the war with his counterparts, Egypt’s ministry said without elaborating.
Israel says 10 soldiers have died in Lebanon since invasion — 5:32 a.m.
By the Associated Press
Israel’s military spokesperson says 10 soldiers have died fighting in Lebanon since the start of the Israeli invasion, including four deaths announced Tuesday.
As of Friday, the military said 261 troops had been injured, 22 seriously, in fighting since the start of the latest war.
UN special rapporteur on Iran decries executions — 5:30 a.m.
By the Associated Press
The United Nations’ special rapporteur on Iran denounced executions being carried out by Tehran.
Mai Sato made the comment on X after two more members of the Iranian exile group Mujahedeen-e-Khalq had been executed Tuesday.
Two others were hanged Monday.
“Given the ongoing internet shutdown, it remains unclear who else has been or is being executed,” she wrote. “What is clear is that executions are being used as a means of suppressing political dissent amid war.”
Israeli soldier dismissed over CNN incident — 5:26 a.m.
By the Associated Press
An Israeli soldier has been dismissed after making “inappropriate comments” to a CNN crew, Israel’s military spokesperson said Tuesday.
The solider’s battalion assaulted and detained the crew in the West Bank last week. CNN said one of the soldiers put producer Cyril Theophilos in a chokehold during the encounter.
It was not clear which soldier was dismissed. There were multiple soldiers filmed by CNN claiming the Israeli-occupied West Bank belonged to them. Other soldiers involved in the incident received reprimands, the spokesperson said.
A formal police investigation was opened into allegations of violence against another soldier, the spokesperson said.
The military’s chief of staff has suspended the battalion from its current deployment.
Read more here.
Chinese ships transit Strait of Hormuz — 4:38 a.m.
By the Associated Press
Three Chinese vessels recently passed through the Strait of Hormuz “through coordination with relevant parties,” Foreign Ministry spokesperson Mao Ning said.
“We appreciate the assistance provided by the relevant parties,” she said without naming them.
She repeated China’s call for an immediate ceasefire, saying the strait is a vital corridor for goods and energy trade.
Gas prices soar past $4 on average for gallon of regular in the US— 3:45 a.m.
By the Associated Press
U.S. gas prices jumped past an average of $4 a gallon for the first time since 2022 as fuel prices continue to soar worldwide.
According to motor club AAA, the national average for a gallon of regular gasoline is now $4.02, over a dollar more than before the war began.
The last time U.S. drivers were collectively paying this much at the pump was nearly four years ago, following Russia’s invasion of Ukraine.
The price is a national average, meaning drivers in some states have been paying well over $4 a gallon for a while now.
Oil steadies and Asian stocks mostly lower — 3:40 a.m.
By the Associated Press
Oil steadied and Asian stocks were mostly lower Tuesday as signs of a de-escalation of the Iran war remained mixed.
Tokyo’s Nikkei 225 was down 1.6% to 51,063.72. South Korea’s Kospi lost 4.3% to 5,052.46.
Hong Kong’s Hang Seng was down 0.3% to 24,678.17, while the Shanghai Composite index fell 0.8% to 3,891.86.
Brent crude futures were less than 0.1% lower at $107.37 a barrel on Tuesday, while benchmark U.S. crude edged up 0.1% to $102.93 per barrel.
Italy and France express ‘concern’ over Lebanon — 3:19 a.m.
By the Associated Press
The defense ministers of Italy and France expressed “deep and profound concern” Tuesday over the deteriorating security in Lebanon.
The joint statement by Guido Crosetto and Catherine Vautrin made particular reference to recent attacks targeting personnel from the United Nations Interim Force in Lebanon.
In a phone conversation Monday they stressed the “unacceptability of such incidents and the increasing risks faced by the personnel deployed in the mission,” the statement said.
The ministers agreed on the strategic importance of UNIFIL, saying Lebanon’s stability constitutes “an indispensable pillar for the balance of the entire Mediterranean basin.”
They confirmed Italy and France will continue operating in close coordination to ensure the safety of international personnel, the protection of peace and support for Lebanese authorities.
Suspected militants attack Pakistan gas pipeline — 2:32 a.m.
By the Associated Press
Suspected militants blew up a local gas pipeline Monday in Pakistan’s southwestern Balochistan province, officials said.
No group immediately claimed responsibility for the attack near Quetta, the capital of the province bordering Iran.
Officials reported the attack Tuesday, saying it disrupted natural gas supplies to regional cities.
The Sui Southern Gas Company said engineers were working to repair the damaged pipeline.
By the Associated Press
Iran held a funeral Tuesday for Rear. Adm. Alireza Tangsiri, the head of Revolutionary Guard’s navy.
An Israeli airstrike killed Tangsiri last week, with Tehran only acknowledging his death Monday.
It showed his casket on a flat bed truck driving through the streets of Bandar Abbas, a crucial port city on the Strait of Hormuz that has seen repeated U.S. airstrikes during the war.
By the Associated Press
A video of a massive explosion shared by U.S. President Donald Trump appears to be of a major strike conducted outside the central Iranian city of Isfahan.
The video shared by Trump without comment early Tuesday includes fiery explosions lighting up the night sky.
The Baluch advocacy group HalVash shared the same video and others from the ground outside of Isfahan. The videos show massive fireballs and secondary explosions common with ammunition igniting in a blaze.
Fire-tracking satellites from NASA suggest the explosions happened near Mount Soffeh, an area believed to have military positions.
Iran has not formally acknowledged the attack.
Isfahan is home to one of three uranium enrichment sites bombed by the U.S. in the 12-day day between Iran and Israel in June. A portion of Iran’s highly enriched uranium is believed to be entombed there, something the U.S. has suggested it could seize with ground forces.
Kuwaiti oil tanker ‘contained’ after attack — 1:32 a.m.
By the Associated Press
Authorities in Dubai said Tuesday morning they “contained” a Kuwaiti oil tanker after it came under attack from Iran.
Officials said there was “no oil leakage and no injuries reported.”
Pakistan’s foreign minister to visit China — 1:28 a.m.
By the Associated Press
Pakistan’s foreign minister left for Beijing on Tuesday for a one-day visit as the country steps up efforts to help end the war in the Middle East.
Ishaq Dar is visiting China at the invitation of his counterpart, Wang Yi, the Foreign Ministry in Islamabad said in a statement without providing additional details.
Dar held consultations over the weekend in Islamabad with top diplomats from Turkey, Egypt and Saudi Arabia.
Dar later said Pakistan would host talks between the United States and Iran in the coming days, though it remains unclear whether they would be direct or indirect.
2 members of Iranian exile group executed — 1:20 a.m.
By the Associated Press
Two more members of the Iranian exile group Mujahedeen-e-Khalq were hanged Tuesday in Iran, state media reported.
The two men were identified as Babak Alipour and Pouya Ghobadi.
Amnesty International has said Tehran’s Revolutionary Court convicted the men on charges of armed rebellion against the state “following a grossly unfair trial in October 2024″ after they were subjected to torture.
Two other MEK members had been hanged Monday over the same case.
Search team boards disabled Thai vessel but does not find missing crew — 1:18 a.m.
By the Associated Press
The operator of a Thai cargo ship struck by a projectile near the Strait of Hormuz said a search team was able to board the vessel but did not locate its missing three crew members.
The Mayuree Naree was disabled after being hit just north of Oman earlier this month.
Precious Shipping Co., Ltd said in a statement to the Stock Exchange of Thailand on Monday that all accessible areas on the Mayuree Naree ship were searched “under challenging conditions, including the presence of fire damage, residual smoke, and flooding in the engine room.” It said the families of the three crew members were notified accordingly.
Images suggest highly enriched uranium was moved to Iran’s Isfahan site before June war — 1:00 a.m.
By the Associated Press
A satellite image taken just before the 12-day war in June between Iran and Israel suggests Tehran transferred a truckload of highly enriched uranium to its nuclear facility at Isfahan.
The image from an Airbus Defense and Space Pléiades Neo satellite shows a truck loaded with 18 blue containers going into a tunnel at the Isfahan Nuclear Technology Center on June 9, 2025. The war began June 13, The United States bombed the Isfahan facility along with two other nuclear sites on June 22.
François Diaz-Maurin, an analyst with the Bulletin of Atomic Scientists, wrote that the truck likely carried 18 secured containers of as much as 534 kilograms (1,177 pounds) of uranium enriched up to 60% purity. That’s a short, technical step to weapons-grade levels of 90%.
“This calculation suggests that Iran could have transferred all of its stockpile of highly enriched uranium to Isfahan via the truck seen in the satellite image,” Diaz-Maurin wrote in his analysis.
The Washington-based Institute for Science and International Security also suggested the vehicle was transferring the highly enriched uranium. The French newspaper Le Monde first reported on the images.
Iran’s foreign minister claims attacks on Gulf Arab states only target US — 12:05 a.m.
By the Associated Press
Iran’s foreign minister early Tuesday insisted that Tehran’s attacks on the Gulf Arab states only target U.S. forces, even after assaults have hit civilian targets throughout the region.
Iranian Foreign Minister Abbas Araghchi’s comments, addressed to Saudi Arabia, come as growing Gulf Arab anger has those states encouraging America to continue to prosecute the war.
“Iran respects the Kingdom of Saudi Arabia and considers it a brotherly nation,” Araghchi wrote on X, sharing a photo purportedly showing damage to an American aircraft at Prince Sultan Air Base in the kingdom. “Our operations are aimed at enemy aggressors who have no respect for Arabs or Iranians, nor can provide any security. ... High time to eject U.S. forces.”

---

### Entire History Back to LLM

The entire context must be sent back to LLM to continue.

**Response API**: build the next `input` by appending `response.output` items + tool results:

```python
# input = original_input + response.output + [function_call_output, ...]
input_with_tools = input_prompt + list(response.output) + tool_outputs
```

| Layer | Chat Completion | Response API |
|---|---|---|
| Original prompt | `messages` list | `input` list |
| LLM decision | `choices[0].message` dict (with `tool_calls`) | `response.output` items (`type=="function_call"`) |
| Tool results | `{"role":"tool","tool_call_id":...,"content":...}` | `{"type":"function_call_output","call_id":...,"output":...}` |

* [Incorporating results into response](https://developers.openai.com/api/docs/guides/function-calling#incorporating-results-into-response)

> After appending the results to your input, send them back to get a final response:
> ```python
> response = client.responses.create(
>     model="gpt-4.1",
>     input=input_messages,
>     tools=tools,
> )
> ```


In [39]:
# Response API: input = original_input + response.output items + function_call_output items
# response.output includes both the function_call items AND any text items from the LLM
# Chat Completion: messages = messages_prompt + [assistant_msg_with_tool_calls] + tool_result_msgs
#
# Key difference: Response API appends response.output items directly (not just the assistant message)
# Ref: https://developers.openai.com/api/docs/guides/function-calling#incorporating-results-into-response
input_with_tool_calls: List[Any] = input_prompt + list(response.output) + tool_outputs

response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_INSTRUCTIONS,
    input=input_with_tool_calls,
    temperature=0,
    max_output_tokens=2048,
    text={
        "format": {
            # ----------------------------------------------------------------------
            # Structured Output (JSON Schema)
            # ----------------------------------------------------------------------
            "type": "json_schema",
            "name": "ResponseStructuredFormat",
            "strict": True,
            "schema": ResponseStructuredFormat.model_json_schema()
        }
    },
    stream=False
)

In [40]:
# Response API: response.output_text (convenience helper for the final text answer)
# Chat Completion: response.choices[0].message.content
print(response.output_text)

{"news":["Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic' - Yahoo","Iran strikes fully loaded oil tanker off Dubai coast as gas reaches $4 per gallon in US. Follow live updates. - The Boston Globe"],"uri":["https://www.yahoo.com/news/articles/trump-calls-irans-current-leaders-034227875.html","https://www.bostonglobe.com/2026/03/31/nation/trump-administration-iran-attacks-updates/"]}


In [41]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "resp_06c103093fd0029b0069d31523357c8193a0c408fa3720ed55",
  "created_at": 1775441187.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "\nYou are a helpful assistant. Use tools when needed.\nVerify the relevance of retrieved results before using them, and apply an intelligent\nintent filter so you keep only the news items that align with the user inquiry.\nOnly report items that are relevant to the user request and supported by the tool output.\nIf the search results are noisy or insufficient, say that and do not infer missing facts.\nRespnse follows the given Python JSON schema where:\n1. news field is the news articies as list.\n2. uri field is the URI of the news article found.\n",
  "metadata": {},
  "model": "gpt-5.2-2025-12-11",
  "object": "response",
  "output": [
    {
      "id": "msg_06c103093fd0029b0069d315238ad4819385cd41ba7bcde249",
      "content": [
        {
          "annotations": [],
          "text": "{\"news\":[\"Trump issues new warni

In [57]:
json.loads(response.output[0].content[0].text)['news']

["Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic' - Yahoo",
 'Iran strikes fully loaded oil tanker off Dubai coast as gas reaches $4 per gallon in US. Follow live updates. - The Boston Globe']

In [25]:
del tool_calls, tool_outputs, input_with_tool_calls, response

---